# The Anatomy of AI · Lecture 11 · Building a Car — the notebook

In the lecture we opened InceptionV1's **car detector**, 4c:447, and found it built from three parts one layer down:
windows on top, a car body, and wheels below. Then we found a dog-head circuit with two mirrored pathways that fight,
and a unit that joins them. Here you rebuild all of it:

1. read the car detector's weights and rank its 512 inputs;
2. see that a weight is a picture;
3. add up the exact sum on one ambulance;
4. break the car detector on 750 car photos;
5. find the mirror twins on 5,900 dog photos and read the weights between them.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab with a T4 GPU, in about ten minutes. It downloads the model (27 MB) and the lecture's own copy
of the ImageNet validation photos (two files of about 350 MB). Section 2 looks at all 50,000 photos once, which takes
about two minutes.

## 0. Setup
Installs (pinned), the plot style, `check()`, the photos and the network (as in Lecture 10's notebook).

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: two parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
import requests
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = f"val-{s}.parquet"
    if not os.path.exists(path):
        t = time.time()
        with requests.get(SHARD.format(s), stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded photo file {s} ({os.path.getsize(path) / 1e6:.0f} MB) in {time.time() - t:.0f} s")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes (256 px square) -> the centre 224 x 224, as a 3 x 224 x 224 tensor in [0, 1]."""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    if (w, h) != (256, 256):
        s = 256 / min(w, h)
        im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        w, h = im.size
    l, t = (w - size) // 2, (h - size) // 2
    im = im.crop((l, t, l + size, t + size))
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network this lecture studies, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

import contextlib
from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name];
    edit={name: fn} replaces a module's output with fn(output) on the way through."""
    def __init__(self, record=(), edit=None):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))
        for name, fn in (edit or {}).items():
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, fn=fn: fn(o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


# InceptionV1 has 1,008 outputs in its own order; PERM puts the 1,000 classes in the standard order (the lecture's
# perm.json, which also fixes two mislabelled outputs), and NAMES gives each class its name.
PERM = torch.tensor(json.loads("[449, 450, 442, 443, 444, 445, 446, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 494, 495, 496, 497, 498, 499, 500, 501, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 473, 475, 476, 474, 477, 478, 479, 480, 481, 482, 483, 484, 485, 486, 487, 488, 489, 490, 491, 492, 493, 601, 602, 603, 604, 605, 606, 607, 608, 609, 610, 611, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 214, 215, 217, 216, 213, 218, 647, 648, 649, 650, 651, 652, 653, 654, 655, 656, 226, 613, 614, 615, 616, 617, 618, 619, 620, 612, 422, 423, 424, 425, 427, 426, 428, 429, 430, 438, 431, 432, 433, 434, 435, 436, 437, 439, 440, 441, 6, 22, 193, 14, 173, 99, 87, 69, 116, 198, 43, 89, 200, 98, 161, 132, 32, 180, 42, 18, 207, 181, 105, 77, 189, 20, 204, 63, 31, 66, 21, 25, 45, 170, 119, 210, 107, 126, 88, 145, 59, 160, 152, 72, 33, 91, 4, 27, 113, 123, 36, 156, 109, 158, 131, 26, 71, 56, 146, 144, 125, 176, 139, 134, 90, 2, 192, 154, 150, 94, 5, 19, 191, 196, 117, 141, 68, 93, 51, 208, 184, 97, 29, 171, 124, 128, 47, 64, 211, 112, 187, 114, 86, 115, 79, 130, 140, 110, 82, 17, 177, 149, 15, 3, 41, 127, 84, 143, 133, 60, 172, 179, 118, 168, 148, 70, 197, 50, 106, 49, 151, 46, 205, 102, 28, 58, 155, 136, 202, 34, 62, 1, 159, 67, 174, 55, 10, 95, 8, 11, 201, 85, 153, 30, 190, 76, 206, 61, 163, 103, 209, 74, 35, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 224, 631, 632, 633, 634, 635, 636, 637, 638, 639, 640, 641, 642, 643, 644, 645, 646, 225, 657, 658, 188, 129, 164, 157, 13, 53, 183, 195, 101, 39, 80, 147, 78, 120, 167, 108, 162, 165, 81, 52, 9, 65, 57, 12, 121, 186, 48, 23, 182, 40, 212, 44, 16, 178, 38, 83, 104, 96, 185, 122, 73, 135, 137, 138, 203, 54, 100, 175, 142, 166, 37, 111, 92, 199, 75, 194, 24, 7, 169, 447, 451, 448, 452, 453, 456, 457, 454, 455, 547, 853, 896, 223, 345, 246, 230, 232, 677, 265, 264, 522, 688, 845, 752, 540, 847, 704, 767, 233, 907, 967, 341, 718, 916, 307, 705, 683, 518, 905, 258, 807, 908, 296, 353, 785, 909, 884, 266, 733, 991, 849, 777, 811, 933, 941, 254, 985, 835, 533, 839, 689, 252, 940, 805, 300, 706, 779, 538, 817, 716, 872, 719, 949, 851, 820, 580, 833, 887, 707, 267, 673, 591, 539, 239, 377, 836, 576, 568, 891, 749, 563, 548, 890, 929, 701, 241, 987, 342, 914, 827, 721, 902, 382, 762, 303, 335, 301, 801, 690, 695, 370, 713, 797, 979, 761, 996, 674, 698, 583, 543, 708, 243, 268, 376, 347, 910, 881, 297, 545, 778, 898, 298, 670, 756, 856, 865, 720, 313, 550, 959, 966, 523, 529, 315, 821, 667, 579, 715, 253, 897, 972, 834, 336, 799, 1000, 662, 512, 346, 262, 316, 879, 660, 808, 796, 305, 235, 279, 919, 995, 356, 309, 784, 261, 712, 934, 299, 256, 348, 671, 757, 280, 971, 567, 955, 275, 792, 276, 236, 337, 911, 227, 684, 725, 703, 517, 581, 895, 249, 375, 840, 505, 551, 750, 573, 350, 344, 554, 369, 787, 696, 730, 562, 802, 924, 293, 525, 980, 659, 592, 748, 269, 961, 963, 294, 560, 770, 773, 582, 918, 892, 814, 228, 374, 988, 371, 687, 238, 546, 270, 244, 867, 973, 894, 508, 536, 697, 532, 818, 917, 782, 977, 763, 338, 339, 781, 984, 598, 922, 946, 302, 717, 509, 661, 866, 875, 920, 880, 271, 251, 871, 829, 290, 272, 764, 686, 869, 277, 824, 854, 691, 852, 260, 255, 728, 511, 599, 284, 588, 585, 595, 755, 915, 552, 699, 354, 351, 520, 513, 333, 870, 937, 295, 506, 921, 826, 564, 584, 504, 759, 685, 352, 877, 942, 994, 306, 527, 257, 679, 843, 732, 842, 850, 883, 783, 789, 575, 926, 722, 281, 596, 931, 901, 888, 841, 565, 245, 983, 372, 693, 964, 731, 381, 378, 857, 923, 285, 855, 314, 788, 838, 566, 373, 769, 556, 702, 542, 534, 572, 846, 939, 862, 976, 273, 860, 571, 863, 537, 927, 286, 570, 965, 668, 578, 694, 219, 541, 310, 663, 997, 876, 519, 760, 753, 586, 952, 751, 938, 355, 809, 521, 962, 221, 729, 510, 587, 379, 589, 559, 800, 709, 832, 950, 259, 380, 868, 747, 590, 776, 943, 549, 726, 557, 507, 288, 278, 960, 222, 986, 577, 925, 822, 858, 515, 234, 676, 237, 600, 775, 274, 593, 804, 263, 680, 340, 530, 998, 724, 528, 516, 514, 287, 957, 311, 692, 247, 794, 526, 574, 535, 810, 681, 828, 837, 945, 569, 561, 531, 304, 250, 978, 675, 786, 944, 970, 989, 903, 758, 555, 308, 780, 664, 710, 312, 594, 700, 282, 711, 289, 283, 766, 825, 291, 242, 597, 678, 882, 349, 765, 727, 544, 220, 292, 334, 666, 874, 990, 969, 558, 790, 682, 343, 936, 665, 524, 928, 317, 231, 906, 669, 958, 819, 795, 772, 502, 899, 912, 904, 935, 831, 503, 672, 951, 815, 723, 248, 240, 714, 553, 930, 791, 932, 861, 774, 803, 754, 813, 844, 771, 793, 974, 968, 873, 768, 975, 993, 885, 734, 736, 737, 738, 739, 740, 741, 742, 743, 744, 735, 745, 746, 318, 229, 319, 320, 321, 322, 323, 324, 325, 326, 816, 999, 953, 864, 806, 948, 830, 900, 798, 947, 859, 823, 361, 992, 359, 365, 368, 366, 363, 364, 367, 360, 362, 954, 848, 982, 330, 357, 358, 331, 327, 328, 332, 886, 913, 956, 893, 878, 812, 981, 329, 889]"), device=DEV)
from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]

In [ ]:
# Every photo's class, for both files (a fast read: just the labels).
ALL_LABELS = np.concatenate([_shard(s).read(columns=["label"]).column("label").to_numpy() for s in (0, 1)]).astype(int)
print(len(ALL_LABELS), "photos")

# The four branches of each block, in channel order: 1x1, 3x3, 5x5, and a pooling branch.
BRANCHES = {"mixed4a": (192, 204, 48, 64), "mixed4b": (160, 224, 64, 64), "mixed4c": (128, 256, 64, 64)}
BRANCH_NAMES = ("1x1", "3x3", "5x5", "pool_reduce")

def branch_of(layer, ch):
    start = 0
    for name, n in zip(BRANCH_NAMES, BRANCHES[layer]):
        if ch < start + n:
            return name, ch - start
        start += n

## 1. Who does the car detector listen to? (Plates I and III)
4c:447 sits in mixed4c's 5 × 5 branch. Its input, all 512 maps of mixed4b, first passes through a 1 × 1 *bottleneck*
(24 maps), then a 5 × 5 filter. Multiply the two weight arrays together and you get one 5 × 5 picture for every
mixed4b map: how much a value there, at each of 25 places round the car detector, pushes it up or down. These are its
**expanded weights**. (The bottleneck's gate is ignored, as Zoom In does.) Rank the 512 inputs by their total weight.

In [ ]:
mods = dict(net.named_modules())

def expanded(layer, unit):
    """A unit's expanded weights from every map of the layer below: [maps below, k, k]."""
    b, i = branch_of(layer, unit)
    W = mods[f"{layer}_{b}_pre_relu_conv"].weight[i].cpu()
    if b in ("1x1", "pool_reduce"):
        return W
    Wb = mods[f"{layer}_{b}_bottleneck_pre_relu_conv"].weight[:, :, 0, 0].cpu()
    return torch.einsum("bhw,bp->phw", W, Wb)

E447 = expanded("mixed4c", 447).numpy()             # 512 x 5 x 5
total = E447.sum((1, 2))
order = np.argsort(-total)
print("strongest inputs:", [(int(u), round(float(total[u]), 3)) for u in order[:10]])
print("most negative:   ", [(int(u), round(float(total[u]), 3)) for u in order[-5:]])
PARTS = {"windows": 237, "car body": 491, "wheels": 373}
check("the top ten inputs of 512", [int(u) for u in order[:10]], [491, 373, 237, 423, 425, 475, 318, 362, 467, 250])
check("car body 4b:491, total weight", float(total[491]), 0.9826, tol=1e-4)
check("wheels 4b:373, total weight", float(total[373]), 0.5977, tol=1e-4)
check("windows 4b:237, total weight", float(total[237]), 0.5974, tol=1e-4)

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.bar(range(512), total[order], width=1.0, color=INK)
for r, (name, c) in enumerate(PARTS.items()):
    k = int(np.where(order == c)[0][0])
    ax.annotate(f"{name} 4b:{c}", (k, total[c]), xytext=(40 + 60 * r, total[c] - 0.05 * r), textcoords="data",
                fontsize=9, arrowprops=dict(arrowstyle="-", color=INK, lw=0.6))
ax.axhline(0, color=INK, lw=0.6); ax.set_xlabel("4c:447's 512 inputs from mixed4b, ranked"); ax.set_ylabel("total weight")
plt.show()

## 2. Meet the car detector (Plate II)
What makes 4c:447 fire hardest, across all 50,000 photos? One pass over the whole validation set, keeping each photo's
strongest value for three mixed4c neurons we need: 447 (the car detector), and 254 and 407 (for section 7).

In [ ]:
KEEP = [447, 254, 407]
peak4c = np.zeros((50000, 3), np.float32)
t = time.time()
with torch.no_grad(), Taps(["mixed4c"]) as T:
    for s in (0, 1):
        f = _shard(s)
        for g in range(f.num_row_groups):
            ims = f.read_row_group(g, columns=["image"]).column("image").to_pylist()
            x = torch.stack([decode(im["bytes"]) for im in ims]).to(DEV)
            net(prep(x))
            n0 = 25000 * s + 100 * g
            peak4c[n0:n0 + len(ims)] = T["mixed4c"][:, KEEP].amax((2, 3)).cpu().numpy()
print(f"50,000 photos in {time.time() - t:.0f} s")
CARS = [407, 436, 468, 511, 609, 627, 656, 661, 717, 751, 817, 734, 675, 864, 654]   # the lecture's 15 car classes
top = np.argsort(-peak4c[:, 0])[:40]
print("4c:447's twelve strongest photos:", [(int(n), NAMES[ALL_LABELS[n]]) for n in top[:12]])
check("4c:447's strongest photo of 50,000", int(top[0]), 25468)
check("its value", float(peak4c[top[0], 0]), 2794.0, tol=0.006 * 2794.0)
check("the lecture's top 40, how many are in ours", len(set(top.tolist()) & set([25468, 26125, 5808, 35787, 8012, 38948, 49888, 41872, 41919, 1177, 3111, 44804, 7843, 2326, 20753, 7373, 18767, 10586, 10880, 31140, 26353, 32189, 22742, 46792, 26906, 41511, 14139, 16944, 21001, 31634, 42415, 48, 20257, 8194, 22512, 28204, 37882, 33520, 4495, 18394])), 40, tol=1)
check("of its top 40, in the 15 car classes", int(np.isin(ALL_LABELS[top], CARS).sum()), 35, tol=1)

In [ ]:
rw = rows(top[:12].tolist())
fig, axs = plt.subplots(2, 6, figsize=(11, 4.2))
for a, n in zip(axs.flat, top[:12]):
    a.imshow(decode(rw[int(n)][0]).permute(1, 2, 0).numpy()); a.axis("off")
    a.set_title(NAMES[ALL_LABELS[n]][:16], fontsize=8)
fig.suptitle("4c:447's twelve strongest photos, of all 50,000", fontsize=11)
plt.show()

## 3. A weight is a picture (Plate IV)
The windows' 5 × 5 picture is not stored anywhere: it is the sum of 24 *paths*, one through each bottleneck map. Each path
is the 5 × 5 filter from that bottleneck map times the windows' weight into it.

In [ ]:
b, i = branch_of("mixed4c", 447)
W5 = mods[f"mixed4c_{b}_pre_relu_conv"].weight[i].cpu().numpy()                  # 24 x 5 x 5
Wb = mods[f"mixed4c_{b}_bottleneck_pre_relu_conv"].weight[:, :, 0, 0].cpu().numpy()  # 24 x 512
paths = W5 * Wb[:, 237][:, None, None]                                             # 24 paths for the windows
fig, axs = plt.subplots(3, 9, figsize=(11, 3.9))
lim = np.abs(paths).max()
for k, a in enumerate(axs.flat):
    if k < 24:
        a.imshow(paths[k], cmap="gray_r", vmin=-lim, vmax=lim); a.set_title(f"path {k}", fontsize=7)
    elif k == 26:
        a.imshow(paths.sum(0), cmap="gray_r"); a.set_title("all 24 added", fontsize=7)
    a.axis("off")
plt.show()
check("the 24 paths add up to the expanded weight (largest difference)", float(np.abs(paths.sum(0) - E447[237]).max()), 0.0, tol=1e-5)

## 4. Windows on top, wheels below (Plate V)
Look at the three parts' pictures. Rows run top to bottom. Add the top two rows, and the bottom two.

In [ ]:
LECTURE = {"windows": [0.944, -0.413], "car body": [0.101, 0.766], "wheels": [-0.304, 0.813]}        # (top two rows, bottom two rows)
fig, axs = plt.subplots(1, 3, figsize=(9, 3.3))
lim = max(np.abs(E447[c]).max() for c in PARTS.values())
for a, (name, c) in zip(axs, PARTS.items()):
    w = E447[c]
    a.imshow(w, cmap="gray_r", vmin=-lim, vmax=lim)
    for (yy, xx), v in np.ndenumerate(w):
        a.text(xx, yy, f"{v:+.2f}".replace("-", "\u2212"), ha="center", va="center", fontsize=7,
               color=PAPER if v > 0.5 * lim else INK)
    a.set_title(f"{name} 4b:{c}", fontsize=10); a.set_xticks([]); a.set_yticks([])
    print(f"{name:9s}: top two rows {w[:2].sum():+.3f}, bottom two rows {w[3:].sum():+.3f}")
    check(f"{name}: top two rows", float(w[:2].sum()), LECTURE[name][0], tol=0.001)
    check(f"{name}: bottom two rows", float(w[3:].sum()), LECTURE[name][1], tol=0.001)
fig.suptitle("dark = pushes the car detector up, light = holds it back", fontsize=10)
plt.show()

The windows push it up at the top and hold it back at the bottom; the wheels do the opposite; the body pushes it up,
most at the bottom. That is Zoom In's caption, word for word.

## 5. The exact sum on a real ambulance (Plates VI–VII)
Photo 25,468, an ambulance, is the car detector's strongest photo of all. At its peak, 4c:447's value before its gate
is an exact sum. It is every mixed4b map's contribution through the 25 places and the 24 bottleneck maps, using
which bottleneck maps were really on for this photo, plus the bias terms.

(The lecture's 2,794.2 came from a GPU that rounds convolutions a little, TF32. On a T4, as on the lecture's own CPU
re-run, it is 2,794.7, so these checks allow ±1.)

In [ ]:
amb_jpg, amb_label = rows([25468])[25468]
amb = decode(amb_jpg)[None].to(DEV)
with torch.no_grad(), Taps(["mixed4b", "mixed4c_5x5_bottleneck_pre_relu_conv", "mixed4c_5x5_pre_relu_conv", "mixed4c"]) as T:
    net(prep(amb))
a4b = T["mixed4b"][0]                                         # 512 x 14 x 14
bott = T["mixed4c_5x5_bottleneck_pre_relu_conv"][0]           # 24 x 14 x 14, before the bottleneck's gate
pre447 = T["mixed4c_5x5_pre_relu_conv"][0, i]
py, px = divmod(int(pre447.argmax()), 14)
Wb_ = mods["mixed4c_5x5_bottleneck_pre_relu_conv"].weight[:, :, 0, 0]
W5_ = mods["mixed4c_5x5_pre_relu_conv"].weight[i]
on = (bott > 0).float()
contrib = torch.zeros(512, device=DEV)
bias = float(mods["mixed4c_5x5_pre_relu_conv"].bias[i])
for dy in range(-2, 3):
    for dx in range(-2, 3):
        yy, xx = py + dy, px + dx
        if 0 <= yy < 14 and 0 <= xx < 14:
            g = W5_[:, dy + 2, dx + 2] * on[:, yy, xx]                   # each bottleneck map's weight, if it was on
            contrib += (g[:, None] * Wb_).sum(0) * a4b[:, yy, xx]
            bias += float((g * mods["mixed4c_5x5_bottleneck_pre_relu_conv"].bias).sum())
co = contrib.cpu().numpy()
print(f"photo 25,468 ({NAMES[amb_label]}): 4c:447 before its gate {float(pre447[py, px]):.1f} at row {py}, column {px}")
print(f"  = sum of 512 contributions {co.sum():.1f} + bias terms {bias:.1f} = {co.sum() + bias:.1f}")
for name, c in PARTS.items():
    print(f"  {name:9s} 4b:{c}: {co[c]:+.1f}")
print(f"  every push up {co[co > 0].sum():+.1f}, every push down {co[co < 0].sum():+.1f}")
check("the photo is an ambulance", NAMES[amb_label], "ambulance")
check("the peak's place", (py, px), (6, 6))
check("4c:447 before its gate", float(pre447[py, px]), 2794.2, tol=1.0)
check("the sum of every part", float(co.sum() + bias), 2794.4, tol=1.0)
check("the sum minus the value before the gate (it is exact)", float(co.sum() + bias) - float(pre447[py, px]), 0.0, tol=0.2)
check("windows' share", float(co[237]), 640.7, tol=0.5)
check("car body's share", float(co[491]), 559.4, tol=0.5)
check("wheels' share", float(co[373]), 500.3, tol=0.5)
check("every push up", float(co[co > 0].sum()), 4106.9, tol=1.0)
check("every push down", float(co[co < 0].sum()), -1313.3, tol=1.0)

In [ ]:
# Where do the parts fire? Their values in the five rows of the car detector's window (its column, top to bottom).
for name, c in PARTS.items():
    col = a4b[c, max(0, py - 2):py + 3, px].cpu().numpy()
    print(f"{name:9s}: {np.round(col, 1).tolist()}")
check("windows, top row", float(a4b[237, py - 2, px]), 1175.7, tol=1)
check("wheels, bottom row", float(a4b[373, py + 2, px]), 1255.7, tol=1)
fig, axs = plt.subplots(1, 4, figsize=(11, 3))
axs[0].imshow(amb[0].permute(1, 2, 0).cpu().numpy()); axs[0].set_title("photo 25,468", fontsize=9)
for a, (name, c) in zip(axs[1:], PARTS.items()):
    a.imshow(a4b[c].cpu().numpy(), cmap="gray_r"); a.set_title(f"{name} 4b:{c}", fontsize=9)
    a.add_patch(plt.Rectangle((px - 2.5, py - 2.5), 5, 5, fill=False, ec=INK, lw=1.5, ls="--"))
for a in axs: a.axis("off")
plt.show()

Windows in the top rows, wheels in the bottom rows: the photo lines up with the weights. Now really switch parts off,
setting their maps to zero and running the network again, and compare with just subtracting their shares.

In [ ]:
state = {"zero": [], "vflip": []}

def edit4b(o):
    if not state["zero"] and not state["vflip"]:
        return o
    o = o.clone()
    for c in state["zero"]:
        o[:, c] = 0
    for c in state["vflip"]:
        o[:, c] = o[:, c].flip(1)                      # this one map, upside down
    return o


def amb_with(zero=(), vflip=()):
    state["zero"], state["vflip"] = list(zero), list(vflip)
    with torch.no_grad(), Taps(["mixed4c_5x5_pre_relu_conv"], edit={"mixed4b": edit4b}) as T:
        net(prep(amb))
    state["zero"], state["vflip"] = [], []
    return float(T["mixed4c_5x5_pre_relu_conv"][0, i, py, px])

full = amb_with()
off = {name: amb_with([c]) for name, c in PARTS.items()}
all3 = amb_with(PARTS.values())
print(f"nothing off {full:.1f}; " + "; ".join(f"{k} off {v:.1f}" for k, v in off.items()) + f"; all three off {all3:.1f} ({100 * all3 / full:.0f}%)")
print(f"subtracting the three shares instead would give {full - co[[237, 491, 373]].sum():.1f}")
check("windows really off", off["windows"], 2125.9, tol=1.5)
check("car body really off", off["car body"], 2149.1, tol=1.5)
check("wheels really off", off["wheels"], 2277.8, tol=1.5)
check("all three really off", all3, 980.5, tol=1.5)
check("by subtracting the shares", full - float(co[[237, 491, 373]].sum()), 1093.8, tol=1.5)

Really switching all three off leaves 35%, less than subtracting their shares predicts (39%): the rest of the network
shifts too, because which bottleneck maps are on changes.

## 6. Can you break it? (Plates IX–X)
The lecture's 750 car photos: all 50 test photos of each of 15 car classes. We keep the half where 4c:447 fires most
(375 photos) and ask how much of its peak it keeps under each change, photo by photo, then take the middle value (the
*median*). The changes:
- switch parts off;
- flip a part's map upside down *inside* the network, so the windows now sit below the wheels;
- turn the whole photo upside down;
- three random other maps, for scale.

In [ ]:
car_ids = np.where(np.isin(ALL_LABELS, CARS))[0].tolist()
car_rows = rows(car_ids)
rng = np.random.default_rng(0)
top10 = [int(u) for u in order[:10]]
rand3 = [sorted(rng.choice([u for u in range(512) if u not in top10], 3, replace=False).tolist()) for _ in range(3)]
check("the 750 car photos", len(car_ids), 750)
check("the three random sets (numpy seed 0)", rand3, [[258, 322, 432], [8, 20, 37], [254, 327, 464]])
P = list(PARTS.values())
CONDS = {"windows off": ([237], []), "body off": ([491], []), "wheels off": ([373], []), "all three off": (P, []),
         "top 10 off": (top10, []), "random off 1": (rand3[0], []), "random off 2": (rand3[1], []), "random off 3": (rand3[2], []),
         "windows flipped": ([], [237]), "body flipped": ([], [491]), "all three flipped": ([], P)}
res = {k: [] for k in ["base", "photo upside down", *CONDS]}
with torch.no_grad(), Taps(["mixed4c"], edit={"mixed4b": edit4b}) as T:
    for s in range(0, len(car_ids), 50):
        x = torch.stack([decode(car_rows[n][0]) for n in car_ids[s:s + 50]]).to(DEV)
        net(prep(x)); res["base"].append(T["mixed4c"][:, 447].amax((1, 2)).cpu())
        net(prep(x.flip(2))); res["photo upside down"].append(T["mixed4c"][:, 447].amax((1, 2)).cpu())
        for k, (z, v) in CONDS.items():
            state["zero"], state["vflip"] = z, v
            net(prep(x)); res[k].append(T["mixed4c"][:, 447].amax((1, 2)).cpu())
        state["zero"], state["vflip"] = [], []
res = {k: torch.cat(v).numpy() for k, v in res.items()}
base = res["base"]; strong = base > np.percentile(base, 50)
keeps = {k: 100 * float(np.median(v[strong] / base[strong])) for k, v in res.items() if k != "base"}
for k, v in keeps.items():
    print(f"  {k:20s} keeps {v:5.1f}%")
check("photos where it fires most", int(strong.sum()), 375)
LECTURE = {"windows off": 83.5, "body off": 76.6, "wheels off": 81.7,
           "all three off": 37.3, "top 10 off": 26.0,
           "random off 1": 101.8, "random off 2": 100.9, "random off 3": 100.4,
           "windows flipped": 85.7, "body flipped": 93.2, "all three flipped": 56.6,
           "photo upside down": 24.4}
for k, v in LECTURE.items():
    check(f"{k}: what it keeps (%)", keeps[k], v, tol=0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
ks = list(keeps)
for j, k in enumerate(ks):
    v = 100 * res[k][strong] / base[strong]
    ax.plot(np.full(len(v), j) + np.random.default_rng(j).uniform(-0.3, 0.3, len(v)), v, ".", ms=1.5, color=INK2)
    ax.plot([j - 0.38, j + 0.38], [keeps[k]] * 2, color=INK, lw=2.5)
ax.set_xticks(range(len(ks)), ks, rotation=40, ha="right", fontsize=8)
ax.axhline(100, color=INK, lw=0.6, ls="--"); ax.set_ylabel("what 4c:447 keeps (%)"); ax.set_ylim(0, 140)
ax.set_title("375 car photos, one dot each; bar = median", fontsize=10)
plt.show()

No single part is essential (each leaves about 80%), but together they are almost two-thirds of the detector. And
*where* they are matters: windows below the wheels costs 14%, and an upside-down car loses three-quarters.

## 7. Mirror twins: a harder problem (Plates XIII–XIV)
A dog's head can face left or right. Zoom In found units in mixed4a for each way, and later units that answer both.
Take all 5,900 dog photos (classes 151–268) and each one mirrored. A unit's *twin* answers the mirrored photo the way
the unit answers the original. A unit that answers both ways the same is *invariant*: its answers to a photo and its
mirror correlate near 1.

In [ ]:
UNITS = {"mixed4a": [222, 234, 308, 325], "mixed4b": [384, 429, 431, 443, 409, 418], "mixed4c": [254, 407]}
dog_ids = np.where((ALL_LABELS >= 151) & (ALL_LABELS <= 268))[0].tolist()
check("dog photos", len(dog_ids), 5900)
dog_rows = rows(dog_ids)
A, Af = {}, {}
t = time.time()
with torch.no_grad(), Taps(list(UNITS)) as T:
    for s in range(0, len(dog_ids), 100):
        x = torch.stack([decode(dog_rows[n][0]) for n in dog_ids[s:s + 100]]).to(DEV)
        for mirrored, D in ((False, A), (True, Af)):
            net(prep(x.flip(3) if mirrored else x))
            for L, us in UNITS.items():
                for u in us:
                    D.setdefault(f"{L}:{u}", []).append(T[L][:, u].amax((1, 2)).cpu())
A = {k: torch.cat(v).numpy() for k, v in A.items()}; Af = {k: torch.cat(v).numpy() for k, v in Af.items()}
print(f"5,900 dog photos, twice, in {time.time() - t:.0f} s")
r = lambda a, b: float(np.corrcoef(a, b)[0, 1])
INV = {"mixed4a:222": 0.129, "mixed4a:234": 0.153, "mixed4a:308": 0.095, "mixed4a:325": 0.66, "mixed4b:384": 0.124, "mixed4b:429": 0.563, "mixed4b:431": -0.011, "mixed4b:443": 0.457, "mixed4b:409": 0.967, "mixed4b:418": 0.95, "mixed4c:254": 0.897, "mixed4c:407": 0.93}
for k, v in INV.items():
    check(f"{k}: answers to a photo and its mirror, correlation", r(A[k], Af[k]), v, tol=0.005)
MIRROR = {"mixed4a:222~mixed4a:308": 0.888, "mixed4a:234~mixed4a:325": 0.759, "mixed4b:384~mixed4b:443": 0.839, "mixed4b:429~mixed4b:431": 0.712}
for k, v in MIRROR.items():
    a, b_ = k.split("~")
    check(f"twins {a} and {b_} (one photo mirrored)", r(A[a], Af[b_]), v, tol=0.005)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8.5, 3.8))
for a, (u, v) in zip(axs, (("mixed4a:222", "mixed4a:308"), ("mixed4a:222", "mixed4a:222"))):
    a.plot(A[u], Af[v], ".", ms=1.2, color=INK, alpha=0.5)
    a.set_xlabel(f"{u} on the photo"); a.set_ylabel(f"{v} on its mirror image")
    a.set_title(f"r = {r(A[u], Af[v]):.2f}", fontsize=10)
plt.tight_layout(); plt.show()

## 8. Two pathways that fight, and the union (Plates XV–XVI)
Group the oriented heads into two sides, each with its mirror twin on the other side: X = {4a:222, 4a:325} and
Y = {4a:234, 4a:308}. Read each next-stage unit's expanded weights from the four.

In [ ]:
E4b = {u: expanded("mixed4b", u).numpy() for u in (384, 429, 431, 443, 409, 418)}
print("            from 4a:222  4a:325  |  4a:234  4a:308   (side X | side Y)")
for u, Eu in E4b.items():
    s_ = {h: float(Eu[h].sum()) for h in (222, 325, 234, 308)}
    print(f"  4b:{u}:      {s_[222]:+.2f}   {s_[325]:+.2f}   |   {s_[234]:+.2f}   {s_[308]:+.2f}")
LECTURE = {"384": {"222": 0.496, "234": -0.638, "308": -0.574, "325": 0.623}, "429": {"222": 0.361, "234": -0.661, "308": -0.589, "325": 0.554}, "431": {"222": -0.498, "234": 0.339, "308": 0.29, "325": -0.515}, "443": {"222": -0.587, "234": 0.706, "308": 0.365, "325": -0.421}, "409": {"222": 0.093, "234": 0.506, "308": 0.096, "325": 0.684}, "418": {"222": -0.153, "234": -0.371, "308": -0.184, "325": -0.324}}
for u in (384, 429, 431, 443):
    for h in (222, 234, 308, 325):
        check(f"4a:{h} -> 4b:{u}, total weight", float(E4b[u][h].sum()), LECTURE[str(u)][str(h)], tol=0.001)

Each side excites its own next stage and holds back the other's: the pathways fight. Now the invariant head 4b:409.

In [ ]:
E409 = E4b[409]; tot409 = E409.sum((1, 2)); o409 = np.argsort(-tot409)
print("4b:409's strongest 4a inputs:", [(int(u), round(float(tot409[u]), 3)) for u in o409[:5]])
check("4b:409's strongest 4a input", int(o409[0]), 325)
check("4a:234's rank among them", int(np.where(o409 == 234)[0][0]) + 1, 4)
for h, (left, right) in {"234": [0.402, -0.043], "325": [0.04, 0.381]}.items():
    w = E409[int(h)]
    print(f"  4a:{h} -> 4b:409: left two columns {w[:, :2].sum():+.3f}, right two {w[:, 3:].sum():+.3f}")
    check(f"4a:{h} -> 4b:409, left two columns", float(w[:, :2].sum()), left, tol=0.001)
    check(f"4a:{h} -> 4b:409, right two columns", float(w[:, 3:].sum()), right, tol=0.001)
E407 = expanded("mixed4c", 407).numpy()
for h, v in {"384": 0.164, "429": 0.131, "431": 0.102, "443": 0.187}.items():
    check(f"4b:{h} -> 4c:407, total weight", float(E407[int(h)].sum()), v, tol=0.001)
fig, axs = plt.subplots(1, 2, figsize=(6, 3))
lim = max(np.abs(E409[234]).max(), np.abs(E409[325]).max())
for a, h in zip(axs, (234, 325)):
    a.imshow(E409[h], cmap="gray_r", vmin=-lim, vmax=lim); a.set_title(f"4a:{h} -> 4b:409", fontsize=10); a.axis("off")
plt.show()

One head from each side, at opposite sides of 4b:409's window: 4a:234 on the left, 4a:325 on the right. Joining two
mirrored cases is how this unit stops caring which way the head faces. Next, 4c:407 adds up all four head-and-neck
units.

## 9. Not every unit fits (Plate XVII)
Zoom In's figure also links 4c:254 as a dog head-and-neck unit. From section 2's pass over all 50,000 photos, compare its
strongest photos with 4c:407's.

In [ ]:
DOG = lambda ids: int(((ALL_LABELS[ids] >= 151) & (ALL_LABELS[ids] <= 268)).sum())
t254 = np.argsort(-peak4c[:, 1])[:40]; t407 = np.argsort(-peak4c[:, 2])[:40]
print("4c:254:", [NAMES[ALL_LABELS[n]] for n in t254[:12]])
print("4c:407:", [NAMES[ALL_LABELS[n]] for n in t407[:12]])
check("4c:254's strongest photo", int(t254[0]), 7837)
check("dogs among 4c:254's top 40", DOG(t254), 0)
check("dogs among 4c:407's top 40", DOG(t407), 25, tol=1)

4c:254 likes drilling platforms, airships and tripods. In our model it is not part of the dog circuit; 4c:407 is.

## Your turn

1. **What holds it back?** Which five mixed4b maps push 4c:447 down hardest, by total weight? *Hint: the end of `order`.*
2. **Two parts off.** On the ambulance, switch off each pair of parts. Which pair matters most? *Hint: `amb_with([...])`.*
3. **Flip the wheels.** Flip only the wheels map upside down inside the network, on the 375 car photos. And mirror the
   whole photo left to right. *Hint: add to `CONDS`, or use `x.flip(3)` for the mirror.*
4. **The other invariant head.** 4b:418 is as invariant as 4b:409. Is it built from the four oriented heads? *Hint:
   `expanded("mixed4b", 418)`, then its strongest 4a inputs.*
5. **A closer pair.** How alike are 4b:429 and 4b:443 when one photo is mirrored? *Hint: `r(A[...], Af[...])`.*

In [ ]:
# Your turn 1

In [ ]:
# Your turn 2

In [ ]:
# Your turn 3

In [ ]:
# Your turn 4

In [ ]:
# Your turn 5

## Solutions

In [ ]:
# 1. What holds it back?
print("most negative inputs:", [(int(u), round(float(total[u]), 3)) for u in order[-5:]])
check("the five most negative inputs", [int(u) for u in order[-5:]], [424, 393, 463, 420, 386])

In [ ]:
# 2. Two parts off
pairs = {"windows + body": [237, 491], "windows + wheels": [237, 373], "body + wheels": [491, 373]}
LECTURE = {"windows + body": 1511.6, "windows + wheels": 1609.2, "body + wheels": 1618.2}
for k, ps in pairs.items():
    v = amb_with(ps)
    print(f"{k} off: {v:.1f}")
    check(f"{k} off", v, LECTURE[k], tol=1.5)
# Windows and body together matter most here.

In [ ]:
# 3. Flip the wheels, and mirror the photo
wf, mir = [], []
with torch.no_grad(), Taps(["mixed4c"], edit={"mixed4b": edit4b}) as T:
    for s in range(0, len(car_ids), 50):
        x = torch.stack([decode(car_rows[n][0]) for n in car_ids[s:s + 50]]).to(DEV)
        state["vflip"] = [373]; net(prep(x)); wf.append(T["mixed4c"][:, 447].amax((1, 2)).cpu())
        state["vflip"] = []; net(prep(x.flip(3))); mir.append(T["mixed4c"][:, 447].amax((1, 2)).cpu())
wf, mir = torch.cat(wf).numpy(), torch.cat(mir).numpy()
check("wheels flipped: what it keeps (%)", 100 * float(np.median(wf[strong] / base[strong])), 81.0, tol=0.5)
check("photo mirrored: what it keeps (%)", 100 * float(np.median(mir[strong] / base[strong])), 99.6, tol=0.5)
# Left-right does not matter to a car detector; up-down does.

In [ ]:
# 4. The other invariant head
E418 = E4b[418]; t418 = E418.sum((1, 2))
print("4b:418's strongest 4a inputs:", [(int(u), round(float(t418[u]), 3)) for u in np.argsort(-t418)[:5]])
print("from the four oriented heads:", {h: round(float(t418[h]), 3) for h in (222, 234, 308, 325)})
check("4b:418's three strongest 4a inputs", [int(u) for u in np.argsort(-t418)[:3]], [290, 383, 374])
check("all four oriented heads hold it back", bool(all(t418[h] < 0 for h in (222, 234, 308, 325))), True)
# It is invariant, but built from other inputs: a more direct pathway.

In [ ]:
# 5. A closer pair
check("4b:429 and 4b:443, one photo mirrored", r(A["mixed4b:429"], Af["mixed4b:443"]), 0.943, tol=0.005)
print(f"whole notebook: {(time.time() - T0) / 60:.1f} minutes")